In [ ]:
ID ~ (|VGS|-|Vtho|)^2 * [2 tanh(|VDS|/(|VGS|-|Vtho|))-tanh^2(|VDS|/(|VGS|-|Vtho|))].

In [18]:
import numpy as np
import json

# Carrega o JSON
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/index_generated.json', 'r') as f:
    index_data = json.load(f)

X_bruto = [] # Conterá [Vgs, Vds]
Y_labels_ponto = [] # Mantém ln(|Id|)

for item in index_data:
    npz_path = item["npz_path"]
    data = np.load(npz_path, allow_pickle=True)
    
    V_model = data["V"].ravel()
    I_model = data["I"].ravel()
    V_fixed = item["fixed_voltage_simulated"] 
    Vtho = item["params"]["VTHO"]
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = data['meta'].tolist()['is_transfer']
    
    for V_exp, I_exp in zip(V_model, I_model):
        if is_transfer:
            V_GS = V_exp
            V_DS = V_fixed
        else:
            V_GS = V_fixed
            V_DS = V_exp
        
        # --- ADAPTAÇÃO: Lendo dados diretamente sem transformações ---
        # Passamos apenas as variáveis independentes brutas
        X_ponto = [V_GS, V_DS]
        
        # Label Engineering para ln(|Id|)
        # I_abs = np.abs(I_exp)
        # Id_min = 1e-30 
        # Y_label = np.log(max(I_abs, Id_min)) 
        ID = (abs(V_GS) - abs(Vtho))**2 * (2 * np.tanh(abs(V_DS) / (abs(V_GS) - abs(Vtho))) - np.tanh(abs(V_DS) / (abs(V_GS) - abs(Vtho)))**2)
        # Y_label sera ID calculado em cada ponto
        Y_label = ID
        
        
        X_bruto.append(X_ponto)
        Y_labels_ponto.append(Y_label)
        
# Conversão Final
X = np.array(X_bruto, dtype=np.float32)
Y = np.array(Y_labels_ponto, dtype=np.float32).reshape(-1, 1)

print(f"Formato de X: {X.shape}") # Resultado esperado: (n_pontos, 2)

Formato de X: (110000, 2)


In [23]:
X.shape # será (N_total_pontos, 5)
Y.shape # será (N_total_pontos, 1)


(110000, 1)

In [24]:
from sklearn.preprocessing import StandardScaler

# Padronizar X (crucial para convergência)
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

## **EXP 1**

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(256, activation='relu', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(128, activation='relu', name='HL2'),
    Dense(64, activation='relu', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=500, batch_size=32, validation_split=0.2)

2026-01-22 09:30:54.292503: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-22 09:30:56.036159: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-22 09:30:59.036787: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Epoch 1/500


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1769085060.517212 2785756 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1769085060.530202 2785756 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 11956.3828 - mae: 41.9275 - val_loss: 220.6371 - val_mae: 9.9200
Epoch 2/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 78.0557 - mae: 6.1473 - val_loss: 20.4285 - val_mae: 3.4854
Epoch 3/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 14.3888 - mae: 2.7665 - val_loss: 14.1110 - val_mae: 2.7757
Epoch 4/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 13.4048 - mae: 2.5140 - val_loss: 4.3872 - val_mae: 1.5952
Epoch 5/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 9.2869 - mae: 2.0944 - val_loss: 7.4217 - val_mae: 2.0309
Epoch 6/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 8.8430 - mae: 1.9993 - val_loss: 8.1617 - val_mae: 2.1690
Epoch 7/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 6.9200 - mae: 1.7362 - val_loss: 19.1232 - val_mae: 3.4561
Epoch 8/500
2750/2750 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 7.8640 - mae: 1.7516 - val_loss: 1.4698 - val_mae: 0.9181
Epoch 9/500
2750/2750 ━━━━━━━━━

In [26]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from typing import Optional, Tuple, Callable
# Importar a função 'Callable' para indicar o tipo de argumento do modelo

def plot_curve_comparison(
    V_data: np.ndarray, 
    I_real: np.ndarray,
    V_fixed: float,
    is_transfer: bool,
    mlp_model: Optional[Tuple[Callable, object]] = None,
    yscale: str = "log", 
    title: Optional[str] = None
):
    """
    Plota a curva real (I_real vs V_data) e opcionalmente a previsão da MLP.

    Parâmetros
    ----------
    V_data : np.ndarray
        Vetor de tensão (eixo X) lido do CSV (Vg para transferência, Vd para saída).
    I_real : np.ndarray
        Vetor de corrente real (eixo Y) lido do CSV.
    V_fixed : float
        Tensão fixa da curva (Vd para transferência, Vg para saída).
    is_transfer : bool
        True se for curva de transferência (Vgs vs Ids), False se for curva de saída (Vds vs Ids).
    mlp_model : tuple (Callable, object) opcional
        Tupla contendo (função_de_preparação_X, modelo_MLP_treinado, scaler_X).
        Permite que a função gere as previsões internamente.
    yscale : str {"log", "linear", "logarithmic"}
        Tipo de escala no eixo Y.
    title : str
        Título opcional do gráfico.
    """

    fig = go.Figure()
    
    # Normaliza a escala Y para checagem
    y_scale = yscale.lower()
    is_log_scale = y_scale in ("log", "logarithmic")
    y_axis_title = "Corrente Absoluta (A)" if is_log_scale else "Corrente (A)"
    
    # -------------------------------
    # PREPARAÇÃO E PLOTAGEM DA CURVA REAL
    # -------------------------------
    
    # Aplicar a correção Logarítmica apenas para o plot, se necessário
    I_plot_real = np.abs(I_real)
    if is_log_scale:
        I_plot_real[I_plot_real <= 0] = 1e-30
    
    fig.add_trace(go.Scatter(
        x=V_data, 
        y=I_plot_real, 
        mode='lines+markers', 
        name="Real (Dados CSV)",
        line=dict(color='blue')
    ))

    # -------------------------------
    # PREVISÃO E PLOTAGEM DA MLP (Se fornecida)
    # -------------------------------
    if mlp_model is not None:
        # Desempacota as ferramentas necessárias
        prepare_X_func, model, scaler_X = mlp_model
        
        # 1. Preparar features X para a curva inteira
        X_test = prepare_X_func(V_data, V_fixed, is_transfer)
        
        # 2. Padronizar X (CRUCIAL: Usar o scaler treinado)
        X_test_scaled = scaler_X.transform(X_test)
        
        # 3. Prever ln(|Id|)
        Y_pred_ln = model.predict(X_test_scaled).ravel()
        
        # 4. Converter para Corrente (|Id|)
        I_pred_abs = np.exp(Y_pred_ln)
        
        # 5. Preparar para Plotagem Logarítmica (se necessário)
        I_plot_pred = I_pred_abs.copy()
        if is_log_scale:
            I_plot_pred[I_plot_pred <= 0] = 1e-30 
        
        fig.add_trace(go.Scatter(
            x=V_data, 
            y=I_plot_pred, 
            mode='lines', 
            name="Previsão MLP",
            line=dict(color='red', dash='dash')
        ))
        
        # Calcula MAE na escala logarítmica para referência
        I_real_ln = np.log(I_plot_real)
        I_pred_ln_for_metric = np.log(I_plot_pred)
        mae = np.mean(np.abs(I_real_ln - I_pred_ln_for_metric))
        
        if title is None:
             title = f"Curva de Teste vs. Previsão MLP (V_fixed={V_fixed}V, MAE_ln={mae:.3f})"
        else:
             title += f" (MAE_ln={mae:.3f})"


    fig.update_layout(
        title=title if title is not None else "Curva Real (CSV)",
        xaxis_title="V_G (V)" if is_transfer else "V_D (V)",
        yaxis_title=y_axis_title,
        yaxis_type="log" if is_log_scale else "linear",
        template="plotly_dark",
        legend_title="Curvas"
    )

    fig.show()
    return

In [27]:
import numpy as np

def prepare_mlp_features(V_data: np.ndarray, V_fixed: float, is_transfer: bool) -> np.ndarray:
    """
    Gera o array de features X para a MLP a partir dos vetores de tensão de uma curva.
    
    Ajustado para o novo cenário: Retorna apenas as variáveis brutas [Vg, Vd].
    """
    
    # V_data é o vetor do eixo X (V_exp). V_fixed é a tensão constante da simulação.
    
    if is_transfer:
        # Transfer: Eixo X é Vg, Tensão fixa é Vds
        V_G = V_data
        V_D = np.full_like(V_data, V_fixed)
    else: 
        # Output: Eixo X é Vds, Tensão fixa é Vg
        V_G = np.full_like(V_data, V_fixed)
        V_D = V_data

    # Construindo a matriz com apenas 2 colunas
    # Isso deve bater com o input_shape=(2,) da sua rede neural
    X_features = np.column_stack([
        V_G,
        V_D
    ])
    
    return X_features

In [28]:
# Carregar json de teste
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json', 'r') as f:
    infer_data = json.load(f)
for item in infer_data:
    npz_path = item["npz_path"]
    csv_path = item["csv_path"]
    v_fixed = item["fixed_voltage_simulated"]
    params = item["params"]
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = 'transfer' in os.path.basename(npz_path).lower()
    
    # Carregar dados do CSV
    df = pd.read_csv(csv_path)
    V_real = df.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
    I_real = df.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    
    # 2. Preparar o pacote de ferramentas para plotagem
    pacote_ferramentas = (prepare_mlp_features, model, scaler_X)
    
    # 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
    plot_curve_comparison(
        V_data=V_real, 
        I_real=I_real*1e6,
        V_fixed=v_fixed,
        is_transfer=is_transfer,
        mlp_model=pacote_ferramentas,
        yscale="log",
        title=None
    )

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step


/tmp/ipykernel_2785756/911494178.py:80: RuntimeWarning:

overflow encountered in exp



## **EXP 2**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(3, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(3, activation='tanh', name='HL2'),
    Dense(3, activation='tanh', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=10000, batch_size=128, validation_split=0.2)

Epoch 1/10000


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 332765.5000 - mae: 402.6479 - val_loss: 437389.1875 - val_mae: 442.6283
Epoch 2/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 330148.5312 - mae: 399.4825 - val_loss: 434840.2500 - val_mae: 440.0854
Epoch 3/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 327893.1875 - mae: 396.8532 - val_loss: 432421.9375 - val_mae: 437.7309
Epoch 4/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 325710.8750 - mae: 394.4216 - val_loss: 430055.2812 - val_mae: 435.6157
Epoch 5/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 323566.0938 - mae: 392.1708 - val_loss: 427718.3750 - val_mae: 433.6520
Epoch 6/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 321445.8750 - mae: 389.9991 - val_loss: 425402.4688 - val_mae: 431.7328
Epoch 7/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 319342.8125 - mae: 387.8498 - val_loss: 423102.1250 - val_mae: 429.8206
Epoch 8/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 317253

In [15]:
# Carregar json de teste
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json', 'r') as f:
    infer_data = json.load(f)
for item in infer_data:
    npz_path = item["npz_path"]
    csv_path = item["csv_path"]
    v_fixed = item["fixed_voltage_simulated"]
    params = item["params"]
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = 'transfer' in os.path.basename(npz_path).lower()
    
    # Carregar dados do CSV
    df = pd.read_csv(csv_path)
    V_real = df.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
    I_real = df.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    
    # 2. Preparar o pacote de ferramentas para plotagem
    pacote_ferramentas = (prepare_mlp_features, model, scaler_X)
    
    # 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
    plot_curve_comparison(
        V_data=V_real, 
        I_real=I_real*1e6,
        V_fixed=v_fixed,
        is_transfer=is_transfer,
        mlp_model=pacote_ferramentas,
        yscale="log",
        title=None
    )

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/stepWARNING:tensorflow:6 out of the last 9 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7bf36078e4d0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


## **EXP 3**

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(4, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(4, activation='tanh', name='HL2'),
    Dense(4, activation='tanh', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=10000, batch_size=128, validation_split=0.2)

Epoch 1/10000


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 9.2906 - mae: 1.9176 - val_loss: 6.3903 - val_mae: 1.5879
Epoch 2/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.9130 - mae: 0.7847 - val_loss: 1.8397 - val_mae: 0.9442
Epoch 3/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5223 - mae: 0.4460 - val_loss: 0.4691 - val_mae: 0.5107
Epoch 4/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1183 - mae: 0.2234 - val_loss: 0.0712 - val_mae: 0.2020
Epoch 5/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0190 - mae: 0.0994 - val_loss: 0.0080 - val_mae: 0.0714
Epoch 6/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0058 - mae: 0.0616 - val_loss: 0.0035 - val_mae: 0.0496
Epoch 7/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0045 - mae: 0.0554 - val_loss: 0.0029 - val_mae: 0.0451
Epoch 8/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0040 - mae: 0.0523 - val_loss: 0.0030 - val_mae: 0.0457
Epoch 9/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 

In [17]:
# Carregar json de teste
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json', 'r') as f:
    infer_data = json.load(f)
for item in infer_data:
    npz_path = item["npz_path"]
    csv_path = item["csv_path"]
    v_fixed = item["fixed_voltage_simulated"]
    params = item["params"]
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = 'transfer' in os.path.basename(npz_path).lower()
    
    # Carregar dados do CSV
    df = pd.read_csv(csv_path)
    V_real = df.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
    I_real = df.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    
    # 2. Preparar o pacote de ferramentas para plotagem
    pacote_ferramentas = (prepare_mlp_features, model, scaler_X)
    
    # 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
    plot_curve_comparison(
        V_data=V_real, 
        I_real=I_real*1e6,
        V_fixed=v_fixed,
        is_transfer=is_transfer,
        mlp_model=pacote_ferramentas,
        yscale="log",
        title=None
    )

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


## **EXP4**

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(10, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(5, activation='tanh', name='HL2'),
    Dense(5, activation='tanh', name='HL3'),
    Dense(5, activation='tanh', name='HL4'),
    
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=2000, batch_size=128, validation_split=0.2)

Epoch 1/2000


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 5.9936 - mae: 1.4247 - val_loss: 3.2442 - val_mae: 1.0733
Epoch 2/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7386 - mae: 0.4303 - val_loss: 0.5000 - val_mae: 0.4153
Epoch 3/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1099 - mae: 0.1998 - val_loss: 0.0573 - val_mae: 0.1485
Epoch 4/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0211 - mae: 0.1135 - val_loss: 0.0060 - val_mae: 0.0632
Epoch 5/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0098 - mae: 0.0811 - val_loss: 0.0020 - val_mae: 0.0358
Epoch 6/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0075 - mae: 0.0682 - val_loss: 0.0018 - val_mae: 0.0336
Epoch 7/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0056 - mae: 0.0581 - val_loss: 0.0016 - val_mae: 0.0356
Epoch 8/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0038 - mae: 0.0478 - val_loss: 0.0012 - val_mae: 0.0291
Epoch 9/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

In [19]:
# Carregar json de teste
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json', 'r') as f:
    infer_data = json.load(f)
for item in infer_data:
    npz_path = item["npz_path"]
    csv_path = item["csv_path"]
    v_fixed = item["fixed_voltage_simulated"]
    params = item["params"]
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = 'transfer' in os.path.basename(npz_path).lower()
    
    # Carregar dados do CSV
    df = pd.read_csv(csv_path)
    V_real = df.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
    I_real = df.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    
    # 2. Preparar o pacote de ferramentas para plotagem
    pacote_ferramentas = (prepare_mlp_features, model, scaler_X)
    
    # 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
    plot_curve_comparison(
        V_data=V_real, 
        I_real=I_real*1e6,
        V_fixed=v_fixed,
        is_transfer=is_transfer,
        mlp_model=pacote_ferramentas,
        yscale="log",
        title=None
    )

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


## **EXP5**

In [6]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from sklearn.preprocessing import StandardScaler
import joblib

# --- 1. NORMALIZAÇÃO ADAPTADA ---
# Escalonador para as entradas [Vg, Vd]
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Escalonador para a saída ln(|Id|) -> CRUCIAL para melhorar o fit
scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y) # Y já deve ser np.log(max(abs(I), 1e-30))

# --- 2. MODELO COM ARQUITETURA APRIMORADA ---
model = Sequential([
    Input(shape=(2,)), # Entrada bruta [Vg, Vd]
    
    # Camadas mais largas e ativação 'silu' (Swish) para melhor não-linearidade
    # Dense(512, activation='silu', name='HL1'),
    Dense(256, activation='silu', name='HL2'),
    Dense(128, activation='silu', name='HL3'),
    # Dense(64, activation='silu', name='HL4'),
    
    # Dropout leve para evitar que o modelo "decore" ruídos do CSV
    Dropout(0.05),
    
    # Saída linear (vai prever o ln(|Id|) escalonado)
    Dense(1, activation='linear', name='Output')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# --- 3. TREINAMENTO ---
history = model.fit(
    X_scaled, Y_scaled, 
    epochs=1000,          # Aumentado para permitir ajuste fino
    batch_size=128, 
    validation_split=0.2,
    verbose=1
)

# --- 4. SALVAR TUDO PARA A INFERÊNCIA ---
model.save("/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/modelo_otft_otimizado.keras")
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_Y, "scaler_Y.pkl")

print("Treinamento concluído e scalers salvos.")

2026-01-12 17:15:33.907992: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-12 17:15:34.188958: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-12 17:15:40.300085: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1768248942.972866  184004 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1768248942.991214  184004 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are install

Epoch 1/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - loss: 0.0340 - mae: 0.1212 - val_loss: 0.0301 - val_mae: 0.1077
Epoch 2/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 0.0132 - mae: 0.0739 - val_loss: 0.0236 - val_mae: 0.0957
Epoch 3/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - loss: 0.0109 - mae: 0.0653 - val_loss: 0.0199 - val_mae: 0.0921
Epoch 4/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.0096 - mae: 0.0618 - val_loss: 0.0170 - val_mae: 0.0824
Epoch 5/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.0082 - mae: 0.0569 - val_loss: 0.0131 - val_mae: 0.0735
Epoch 6/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 0.0065 - mae: 0.0514 - val_loss: 0.0090 - val_mae: 0.0680
Epoch 7/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 0.0048 - mae: 0.0452 - val_loss: 0.0046 - val_mae: 0.0403
Epoch 8/1000
688/688 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.0032 - mae: 0.0384 - val_loss: 0.0020 - val_mae: 0.0300
Epoch 9/1000
688/688 ━━━━━━━━━

In [5]:
def inference_with_y_scaler(v_data, v_fixed, is_transfer, model, scaler_X, scaler_Y):
    # 1. Preparar e Escalar X
    x_raw = prepare_mlp_features(v_data, v_fixed, is_transfer)
    x_scaled = scaler_X.transform(x_raw)
    # 2. Predição (Resultado ainda está na escala do scaler_Y)
    y_pred_scaled = model.predict(x_scaled, verbose=0)
    
    # 3. VOLTAR PARA O ln(|Id|) REAL
    y_pred_ln = scaler_Y.inverse_transform(y_pred_scaled).ravel()
    
    # 4. VOLTAR PARA AMPERES
    i_pred = np.exp(y_pred_ln)
    
    return i_pred

In [15]:
# # Ferramentas necessárias para a inferência completa
# # Nota: Adicionamos o scaler_Y aqui
# mlp_tools = (prepare_mlp_features, model, scaler_X, scaler_Y)

In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import Optional, Tuple, Callable

def plot_curve(
    V_data: np.ndarray, 
    I_real: np.ndarray,
    V_fixed: float,
    is_transfer: bool,
    mlp_model: Optional[Tuple[Callable, object, object, object]] = None,
    yscale: str = "log", 
    title: Optional[str] = None
):
    """
    Plota a curva real e a previsão da MLP, considerando a desnormalização do Target (Y).

    mlp_model : tuple (prepare_X_func, model, scaler_X, scaler_Y)
    """

    fig = go.Figure()
    
    y_scale = yscale.lower()
    is_log_scale = y_scale in ("log", "logarithmic")
    y_axis_title = "Corrente Absoluta (A)" if is_log_scale else "Corrente (A)"
    
    # --- CURVA REAL ---
    I_plot_real = np.abs(I_real)
    if is_log_scale:
        I_plot_real[I_plot_real <= 0] = 1e-30
    
    fig.add_trace(go.Scatter(
        x=V_data, 
        y=I_plot_real, 
        mode='lines+markers', 
        name="Real (Dados CSV)",
        # marker=dict(symbol='circle-open'),
        line=dict(color='blue')
    ))

    # --- PREVISÃO MLP ---
    if mlp_model is not None:
        # 0. Desempacota as 4 ferramentas (X e Y agora têm scalers)
        prepare_X_func, model, scaler_X, scaler_Y = mlp_model
        
        # 1. Preparar features X (2 colunas: Vg, Vd)
        X_test = prepare_X_func(V_data, V_fixed, is_transfer)
        
        # 2. Padronizar X
        X_test_scaled = scaler_X.transform(X_test)
        
        # 3. Prever Y (O resultado sai escalonado entre ~ -1 e 1)
        Y_pred_scaled = model.predict(X_test_scaled, verbose=0)
        
        # 4. DESNORMALIZAR Y (Voltar para a escala do ln(|Id|))
        # O reshape(-1, 1) é necessário para o scaler do scikit-learn
        Y_pred_ln = scaler_Y.inverse_transform(Y_pred_scaled.reshape(-1, 1)).ravel()
        
        # 5. Converter de Logaritmo para Corrente Linear
        I_pred_abs = np.exp(Y_pred_ln)
        
        # 6. Preparar para Plotagem
        I_plot_pred = I_pred_abs.copy()
        if is_log_scale:
            I_plot_pred[I_plot_pred <= 0] = 1e-30 
        
        fig.add_trace(go.Scatter(
            x=V_data, 
            y=I_plot_pred, 
            mode='lines', 
            name="Previsão MLP (Optimized)",
            line=dict(color='red', width=3, dash='dash')
        ))
        
        # Métrica de Erro: MAE no domínio logarítmico
        # Usamos o real em log para comparar com o Y_pred_ln desnormalizado
        I_real_ln = np.log(np.maximum(I_plot_real, 1e-30))
        mae_ln = np.mean(np.abs(I_real_ln - Y_pred_ln))
        
        suffix = f" (MAE_ln={mae_ln:.4f})"
        title = (title if title else f"V_fixed={V_fixed}V") + suffix


    fig.update_layout(
        title=title,
        xaxis_title="V_G (V)" if is_transfer else "V_D (V)",
        yaxis_title=y_axis_title,
        yaxis_type="log" if is_log_scale else "linear",
        template="plotly_dark",
        legend_title="Curvas"
    )

    fig.show()

In [24]:
import json
import numpy as np
import pandas as pd

# 1. Escolha uma curva do seu JSON para testar
json_path = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
with open(json_path, 'r') as f:
    index_data = json.load(f)

# Vamos testar a primeira curva (índice 0)
item = index_data[0]
v_fixed = item["fixed_voltage_simulated"]
is_transfer = True # Ajuste conforme o tipo da curva

# 2. CORREÇÃO: Carregar dados de um arquivo .CSV
# Usamos pandas para ler o arquivo de texto
df_real = pd.read_csv(item["csv_path"])

# Extraímos as colunas (ajuste o nome se o seu CSV tiver cabeçalho, 
# caso contrário, usamos .iloc para pegar por posição)
V_real = df_real.iloc[:, 0].values  # Primeira coluna: Tensão
I_real = df_real.iloc[:, 1].values  # Segunda coluna: Corrente


In [22]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model
import joblib

PATH_MODEL = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/modelo_otft_otimizado.keras'
trained_model = load_model(PATH_MODEL)
scaler_X = joblib.load("scaler_X.pkl")
scaler_Y = joblib.load("scaler_Y.pkl")

In [25]:

# 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
pacote_ferramentas = (prepare_mlp_features, trained_model, scaler_X, scaler_Y)

plot_curve(
    V_data=V_real, 
    I_real=I_real*1e5,
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    mlp_model=pacote_ferramentas,
    yscale="log",
    title="Teste de Inferência - OTFT"
)